## 最適化問題の種類を知ろう
数理最適化問題には、その解法から
- 線形最適化
- 非線形最適化
- 組み合わせ最適化

の３ Pattern に分類される。

### 線形最適化
- 目的関数が線形（１次関数）で表現される。
- 古くから研究されている。
- 線形最適化問題を解くことそのもを「線形計画」と呼ぶ。
  問題を内定化する際に、線形最適化問題として表現し、線形最適化問題としての解き方を行なうことで（計算時間を度外視すれば）どんな問題でも解くことができる。
- 要素を表の形で整理し、目的関数と制約条件を足し算の形で表現することで様々な場合について応用できる。
- 制約条件を考慮しながら目的関数を最小化していくという Process は、線形計画として最適化問題を解くうえで共通しており、細かい Program を実装せずとも **"Solver"** と呼ばれる Tool によって解くことができる。

### 非線形最適化
目的関数が２次関数などの線形（直線）でない形（すなわち非線形の形）の関数で表現される。
目的関数が非線形の形で表せれるものは、社会の中で多く見られる。

### 組み合わせ最適化
さまざまな組み合わせの中から最適なものを選択するもの。
その多くは目的関数を工夫することにより、線形最適化問題として解くことができる。

## Solver を利用して線形最適化問題を解いてみよう

#### ex).
スーパーマーケットで売られている食料品の中から、何をどれだけ購入すれば栄養素を最も安く入手できるか
３種類の食料品と４種類の栄養素の組み合わせを考えていく。

In [3]:
import numpy as np
import pandas as pd
from itertools import product
from pulp import LpVariable, lpSum, value
from ortoolpy import model_min, addvars, addvals
from IPython.display import display

# Data読み込み
df_n = pd.read_csv('data/Chapter5/nutrition.csv', index_col='食料品')
df_p = pd.read_csv('data/Chapter5/price.csv')
print("食料品と栄養素の関係")
display(df_n)
print("食料品の価格")
display(df_p)

# 初期設定 #
np.random.seed(1)
nx = len(df_n.index)
nn = len(df_n.columns)
pr = list(range(nx))

# 数理 Model 作成 #
m1 = model_min()
# 目的関数
v1 = {(i): LpVariable('v%d' % (i), cat='Integer', lowBound=0) for i in pr}
# 制約条件
m1 += lpSum(df_p.iloc[0][i] * v1[i] for i in pr)
for j in range(nn):
    m1 += lpSum(v1[i] * df_n.iloc[i][j] for i in range(nx)) >= 100
m1.solve()

# 総Cost計算 #
print("最適解")
total_cost = 0
for k, x in v1.items():
    i = k
    print(df_n.index[i], "の個数:", int(value(x)), "個")
    total_cost += df_p.iloc[0][i] * value(x)

print("総Cost:", int(total_cost), "円")

食料品と栄養素の関係


,栄養素１,栄養素２,栄養素３,栄養素４
食料品,,,,
食料品１,0,0,90,10
食料品２,10,50,40,70
食料品３,80,0,0,0


食料品の価格


,食料品１,食料品２,食料品３
0,1000,150,300


最適解
食料品１ の個数: 0 個
食料品２ の個数: 3 個
食料品３ の個数: 1 個
総Cost: 750 円
